In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier

In [2]:
train_df = pd.read_csv("../data/m_tournament_training_dataset_advanced.csv")
submission_df = pd.read_csv("../data/SampleSubmissionStage1.csv")

In [3]:
selected_cols = [
    "Season",
    "Target",
    "SeedNumDiff",
    "RankingDiff",
    "MarginDiff",
    "NetRatingDiff",
    "OffEffDiff",
    "DefEffDiff",
    "WinPctDiff",
    "OffDefGap",
    "DominanceScore",
    "NetRating_Margin_Interaction",
    "Margin_Ranking_Interaction",
    "TurnoverMarginDiff",
    "ReboundPctDiff"
]

In [4]:
final_rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    min_samples_split=15,
    random_state=42,
    n_jobs=-1
)

In [5]:
X_train = train_df[selected_cols].copy()
y_train = train_df["Target"].copy()

final_rf_model.fit(X_train, y_train)

print("Final model trained.")
print("Training shape:", X_train.shape)

Final model trained.
Training shape: (2898, 15)


In [6]:
def parse_kaggle_id(id_str):
    season, team1, team2 = id_str.split("_")
    return int(season), int(team1), int(team2)

In [7]:
submission_df[["Season", "Team1ID", "Team2ID"]] = submission_df["ID"].apply(
    lambda x: pd.Series(parse_kaggle_id(x))
)

In [8]:
print(submission_df.head())

               ID  Pred  Season  Team1ID  Team2ID
0  2022_1101_1102   0.5    2022     1101     1102
1  2022_1101_1103   0.5    2022     1101     1103
2  2022_1101_1104   0.5    2022     1101     1104
3  2022_1101_1105   0.5    2022     1101     1105
4  2022_1101_1106   0.5    2022     1101     1106


In [9]:
submission_features_df = pd.read_csv("../data/m_submission_matchups_advanced.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data/m_submission_matchups_advanced.csv'

In [ ]:
submission_features_df = submission_features_df[["Season", "Team1ID", "Team2ID"] + selected_cols].copy()

In [ ]:
print("Submission features shape:", submission_features_df.shape)
print(submission_features_df.head())

In [ ]:
submission_ready_df = submission_df.merge(
    submission_features_df,
    on=["Season", "Team1ID", "Team2ID"],
    how="left"
)

In [ ]:
print("Merged submission shape:", submission_ready_df.shape)

In [ ]:
missing_counts = submission_ready_df[selected_cols].isna().sum()
print("Missing values by feature:")
print(missing_counts[missing_counts > 0])

In [ ]:
submission_ready_df[selected_cols] = submission_ready_df[selected_cols].fillna(0)

In [ ]:
X_submission = submission_ready_df[selected_cols].copy()

In [ ]:
submission_ready_df["Pred"] = final_rf_model.predict_proba(X_submission)[:, 1]

In [ ]:
print(submission_ready_df[["ID", "Pred"]].head())

In [ ]:
submission_ready_df["Pred"] = submission_ready_df["Pred"].clip(0.025, 0.975)

In [ ]:
final_submission = submission_ready_df[["ID", "Pred"]].copy()

In [ ]:
output_path = "../data/final_kaggle_submission_rf.csv"
final_submission.to_csv(output_path, index=False)

In [ ]:
print(f"Saved submission file to: {output_path}")
print(final_submission.head())
print(final_submission.shape)